In [ ]:
# installing dependencies
#!pip install -q transformers librosa wandb scikit-learn tqdm
'''
This notebook was used to train the MERT model on Google Colab.
'''

In [ ]:
import os
import zipfile

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/MyDrive/dlgenai-mert"
MODEL_DIR  = os.path.join(DRIVE_ROOT, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

# genres_stems is extracted to local /content for fast file I/O
GENRES_STEMS_ZIP = os.path.join(DRIVE_ROOT, "genres_stems.zip")
GENRES_STEMS_DIR = "/content/genres_stems"

if not os.path.exists(GENRES_STEMS_DIR):
    print("Extracting genres_stems...")
    with zipfile.ZipFile(GENRES_STEMS_ZIP, "r") as zf:
        zf.extractall("/content")
    print("Done.")

Extracting genres_stems...
Done.


In [ ]:
# ESC-50 is public and downloaded directly
ESC50_AUDIO_DIR = "/content/ESC-50-master/audio"
if not os.path.exists(ESC50_AUDIO_DIR):
    print("Downloading ESC-50...")
    os.system("wget -q https://github.com/karolpiczak/ESC-50/archive/master.zip -O /content/ESC-50.zip")
    with zipfile.ZipFile("/content/ESC-50.zip", "r") as zf:
        zf.extractall("/content")
    print("Done.")

Done.


In [7]:
# imports and configuration
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoModel, Wav2Vec2FeatureExtractor
import librosa
import wandb
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [ ]:
GENRES = ["blues", "classical", "country", "disco", "hiphop",
                "jazz", "metal", "pop", "reggae", "rock"]
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

MERT_MODEL_ID = "m-a-p/MERT-v1-95M"
SR_MERT = 24000
DURATION = 10.0
TARGET_LEN = int(SR_MERT * DURATION)
SAMPLES_PER_GENRE = 350
VAL_SPLIT  = 0.2
DATA_SEED  = 67
TRAINING_SEED = 1234

GAIN_MIN = 0.5
GAIN_MAX = 1.5
SNR_MIN_DB = 5.0
SNR_MAX_DB = 20.0
STEMS = ["drums", "vocals", "bass", "other"]

In [ ]:
# Training configs
BATCH_SIZE = 8
WEIGHT_DECAY= 0.01
P1_EPOCHS = 5
P1_LR = 1e-3
P2_EPOCHS = 10
P2_LR_HEAD = 1e-4
P2_LR_TRUNK= 1e-5
N_UNFREEZE = 2

In [ ]:
# local cache for precomputed waveforms (re-generated each session, fast)
CACHE_DIR = "/content/mert_waveform_cache"

In [ ]:
# augmentation utilities
def _pad_or_trim(y, length):
    if len(y) >= length:
        return y[:length]
    return np.pad(y, (0, length - len(y)))

def make_augmented_mix(genre, data_root, sr, duration, rng):
    """
    Builds one training sample via cross-song stem mixing.
    Each stem is drawn from an independently chosen song of the same genre,
    matching the cross-song recombination used to construct test mashups.
    """
    target_len = int(sr * duration)
    genre_dir  = os.path.join(data_root, genre)
    songs      = sorted(os.listdir(genre_dir))
    mix        = np.zeros(target_len, dtype=np.float32)

    for stem in STEMS:
        song = rng.choice(songs)
        path = os.path.join(genre_dir, song, f"{stem}.wav")
        try:
            y, _ = librosa.load(path, sr=sr)
            if len(y) > target_len:
                start = rng.randint(0, len(y) - target_len)
                y = y[start: start + target_len]
            else:
                y = _pad_or_trim(y, target_len)
            mix += y * rng.uniform(GAIN_MIN, GAIN_MAX)
        except Exception:
            pass

    noise_files = [f for f in os.listdir(ESC50_AUDIO_DIR) if f.endswith(".wav")]
    try:
        noise, _ = librosa.load(
            os.path.join(ESC50_AUDIO_DIR, rng.choice(noise_files)), sr=sr
        )
        noise      = _pad_or_trim(noise, target_len)
        snr_db     = rng.uniform(SNR_MIN_DB, SNR_MAX_DB)
        sig_rms    = np.sqrt(np.mean(mix   ** 2)) + 1e-9
        noise_rms  = np.sqrt(np.mean(noise ** 2)) + 1e-9
        target_rms = sig_rms / (10 ** (snr_db / 20.0))
        mix        = mix + noise * (target_rms / noise_rms)
    except Exception:
        pass

    peak = np.max(np.abs(mix))
    if peak > 0:
        mix /= (peak + 1e-9)
    return mix

In [ ]:
# precompute augmented waveforms
def precompute_waveforms(data_root, cache_dir, genres,
                         sr, duration, samples_per_genre, val_split, seed):
    """
    Generates augmented waveform mixes and saves them as float32 tensors.
    This eliminates repeated librosa loading during training.
    """
    all_items = [(g, i) for g in genres for i in range(samples_per_genre)]
    train_items, val_items = train_test_split(
        all_items, test_size=val_split, random_state=seed,
        stratify=[g for g, _ in all_items]
    )

    for split, items in [("train", train_items), ("val", val_items)]:
        split_dir = os.path.join(cache_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        for pos, (genre, idx) in enumerate(tqdm(items, desc=f"Precomputing [{split}]")):
            out_path = os.path.join(split_dir, f"{split}_{pos:05d}.pt")
            if os.path.exists(out_path):
                continue
            rng = random.Random(seed + pos)
            mix = make_augmented_mix(genre, data_root, sr, duration, rng)
            torch.save({
                "waveform": torch.tensor(mix, dtype=torch.float32),
                "label":    GENRE_TO_IDX[genre]
            }, out_path)


precompute_waveforms(
    data_root=GENRES_STEMS_DIR, cache_dir=CACHE_DIR, genres=GENRES,
    sr=SR_MERT, duration=DURATION, samples_per_genre=SAMPLES_PER_GENRE,
    val_split=VAL_SPLIT, seed=DATA_SEED
)

Precomputing [val]: 100%|██████████| 700/700 [03:14<00:00,  3.60it/s]


In [ ]:
class PrecomputedWaveformDataset(Dataset):
    """Loads precomputed waveform tensors from disk."""
    def __init__(self, cache_dir, split):
        split_dir  = os.path.join(cache_dir, split)
        self.files = sorted(
            os.path.join(split_dir, f)
            for f in os.listdir(split_dir)
            if f.endswith(".pt")
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        item = torch.load(self.files[idx], weights_only=True)
        return item["waveform"], item["label"]

In [ ]:
train_dataset = PrecomputedWaveformDataset(CACHE_DIR, "train")
val_dataset   = PrecomputedWaveformDataset(CACHE_DIR, "val")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Train: 2800 | Val: 700


In [ ]:
class MERTClassifier(nn.Module):
    """
    MERT backbone with a linear classification head.
    The backbone's last hidden state is mean-pooled over the time dimension
    before being passed to the classifier.
    """
    def __init__(self, model_id, num_classes=10):
        super().__init__()
        self.backbone   = AutoModel.from_pretrained(model_id, trust_remote_code=True)
        hidden_size     = self.backbone.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_values):
        outputs = self.backbone(input_values=input_values, output_hidden_states=False)
        pooled  = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(pooled)

In [ ]:
# training utilities
criterion = nn.CrossEntropyLoss()

def normalise_waveforms(waveforms):
    """Applies per-sample mean-std normalisation, matching Wav2Vec2FeatureExtractor behaviour."""
    mean = waveforms.mean(dim=-1, keepdim=True)
    std = waveforms.std(dim=-1, keepdim=True).clamp(min=1e-7)
    return (waveforms - mean) / std


def run_epoch(model, loader, criterion, optimizer, training, device, desc=""):
    model.train() if training else model.eval()
    total_loss, preds, labels_all = 0.0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for waveforms, labels in tqdm(loader, desc=desc):
            waveforms    = normalise_waveforms(waveforms).to(device)
            labels       = labels.to(device)
            if training:
                optimizer.zero_grad()
            logits = model(waveforms)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * waveforms.size(0)
            preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
            labels_all.extend(labels.detach().cpu().numpy())

    n = len(loader.dataset)
    avg_loss = total_loss / n
    f1 = f1_score(labels_all, preds, average="macro")
    acc = accuracy_score(labels_all, preds)
    return avg_loss, f1, acc

In [ ]:
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [ ]:
# phase 1 training (classifier head only, backbone frozen)
model = MERTClassifier(MERT_MODEL_ID, num_classes=10).to(DEVICE)

config.json: 0.00B [00:00, ?B/s]

configuration_MERT.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- configuration_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_MERT.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- modeling_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

In [ ]:
for param in model.backbone.parameters():
    param.requires_grad = False

trainable_p1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 trainable parameters: {trainable_p1:,}")

optimizer_p1 = optim.AdamW(
    model.classifier.parameters(), lr=P1_LR, weight_decay=WEIGHT_DECAY
)

Phase 1 trainable parameters: 7,690


In [ ]:
wandb.init(
    project="21f3002298-dl-genai-project",
    name="mert-src-phase1",
    config={
        "model":             MERT_MODEL_ID,
        "phase":             1,
        "epochs":            P1_EPOCHS,
        "lr":                P1_LR,
        "batch_size":        BATCH_SIZE,
        "samples_per_genre": SAMPLES_PER_GENRE,
        "sr":                SR_MERT,
        "duration":          DURATION,
    }
)

In [ ]:
best_val_f1_p1 = 0.0
for epoch in range(P1_EPOCHS):
    train_loss, train_f1, train_acc = run_epoch(
        model, train_loader, criterion, optimizer_p1, training=True,
        device=DEVICE, desc=f"[P1] Epoch {epoch+1}/{P1_EPOCHS} [Train]"
    )
    val_loss, val_f1, val_acc = run_epoch(
        model, val_loader, criterion, optimizer_p1, training=False,
        device=DEVICE, desc=f"[P1] Epoch {epoch+1}/{P1_EPOCHS} [Val]"
    )
    torch.cuda.empty_cache()

    wandb.log({
        "epoch":      epoch + 1,
        "train_loss": train_loss, "train_f1": train_f1, "train_acc": train_acc,
        "val_loss":   val_loss,   "val_f1":   val_f1,   "val_acc":   val_acc,
    })
    print(f"[P1] Epoch {epoch+1}/{P1_EPOCHS}: train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1_p1:
        best_val_f1_p1 = val_f1
        torch.save(model.state_dict(),
                   os.path.join(MODEL_DIR, "mert_src_phase1_best.pth"))
        print(f"  Saved best model (val_f1={val_f1:.4f})")

[P1] Epoch 1/5 [Val]: 100%|██████████| 88/88 [00:57<00:00,  1.53it/s]


[P1] Epoch 1/5: train_f1=0.4946 | val_f1=0.5410
  Saved best model (val_f1=0.5410)


[P1] Epoch 2/5 [Val]: 100%|██████████| 88/88 [00:57<00:00,  1.53it/s]


[P1] Epoch 2/5: train_f1=0.6854 | val_f1=0.5903
  Saved best model (val_f1=0.5903)


[P1] Epoch 3/5 [Val]: 100%|██████████| 88/88 [00:57<00:00,  1.52it/s]


[P1] Epoch 3/5: train_f1=0.7495 | val_f1=0.5947
  Saved best model (val_f1=0.5947)


[P1] Epoch 4/5 [Val]: 100%|██████████| 88/88 [00:57<00:00,  1.52it/s]


[P1] Epoch 4/5: train_f1=0.7800 | val_f1=0.6419
  Saved best model (val_f1=0.6419)


[P1] Epoch 5/5 [Val]: 100%|██████████| 88/88 [00:57<00:00,  1.52it/s]


[P1] Epoch 5/5: train_f1=0.8070 | val_f1=0.6538
  Saved best model (val_f1=0.6538)


In [ ]:
print(type(model.backbone.encoder.layers))
print(f"Total encoder layers: {len(model.backbone.encoder.layers)}")

<class 'torch.nn.modules.container.ModuleList'>
Total encoder layers: 12


In [ ]:
# Phase 2 training (last N encoder blocks + head unfrozen)
model.load_state_dict(torch.load(
    os.path.join(MODEL_DIR, "mert_src_phase1_best.pth"),
    map_location=DEVICE, weights_only=True
))

for param in model.backbone.parameters():
    param.requires_grad = False

encoder_blocks = model.backbone.encoder.layers
for block in encoder_blocks[-N_UNFREEZE:]:
    for param in block.parameters():
        param.requires_grad = True

trainable_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2 trainable parameters: {trainable_p2:,}")

Phase 2 trainable parameters: 14,183,434


In [ ]:
head_params  = list(model.classifier.parameters())
trunk_params = [p for block in encoder_blocks[-N_UNFREEZE:]
                for p in block.parameters()]

optimizer_p2 = optim.AdamW([
    {"params": head_params,  "lr": P2_LR_HEAD},
    {"params": trunk_params, "lr": P2_LR_TRUNK},
], weight_decay=WEIGHT_DECAY)
scheduler_p2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=P2_EPOCHS)

In [ ]:
wandb.init(
    project="21f3002298-dl-genai-project",
    name="mert-src-phase2",
    config={
        "model":             MERT_MODEL_ID,
        "phase":             2,
        "epochs":            P2_EPOCHS,
        "lr_head":           P2_LR_HEAD,
        "lr_trunk":          P2_LR_TRUNK,
        "n_unfreeze_blocks": N_UNFREEZE,
        "batch_size":        BATCH_SIZE,
        "samples_per_genre": SAMPLES_PER_GENRE,
    }
)

epoch,▁▃▅▆█
train_acc,▁▅▇▇█
train_f1,▁▅▇▇█
train_loss,█▄▃▂▁
val_acc,▁▄▄██
val_f1,▁▄▄▇█
val_loss,█▅▃▂▁
epoch,5
train_acc,0.80893
train_f1,0.80698
train_loss,0.78799


In [ ]:
best_val_f1_p2 = 0.0
for epoch in range(P2_EPOCHS):
    train_loss, train_f1, train_acc = run_epoch(
        model, train_loader, criterion, optimizer_p2, training=True,
        device=DEVICE, desc=f"[P2] Epoch {epoch+1}/{P2_EPOCHS} [Train]"
    )
    val_loss, val_f1, val_acc = run_epoch(
        model, val_loader, criterion, optimizer_p2, training=False,
        device=DEVICE, desc=f"[P2] Epoch {epoch+1}/{P2_EPOCHS} [Val]"
    )
    scheduler_p2.step()
    torch.cuda.empty_cache()

    wandb.log({
        "epoch":      epoch + 1,
        "train_loss": train_loss, "train_f1": train_f1, "train_acc": train_acc,
        "val_loss":   val_loss,   "val_f1":   val_f1,   "val_acc":   val_acc,
        "lr_head":    optimizer_p2.param_groups[0]["lr"],
        "lr_trunk":   optimizer_p2.param_groups[1]["lr"],
    })
    print(f"[P2] Epoch {epoch+1}/{P2_EPOCHS}: train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1_p2:
        best_val_f1_p2 = val_f1
        torch.save(model.state_dict(),
                   os.path.join(MODEL_DIR, "mert_src_phase2_best.pth"))
        print(f"  Saved best model (val_f1={val_f1:.4f})")

[P2] Epoch 1/10 [Val]: 100%|██████████| 88/88 [00:57<00:00,  1.52it/s]


[P2] Epoch 1/10: train_f1=0.8298 | val_f1=0.6492
  Saved best model (val_f1=0.6492)


[P2] Epoch 2/10 [Val]: 100%|██████████| 88/88 [00:57<00:00,  1.52it/s]


[P2] Epoch 2/10: train_f1=0.8745 | val_f1=0.6599
  Saved best model (val_f1=0.6599)


[P2] Epoch 3/10 [Val]: 100%|██████████| 88/88 [00:57<00:00,  1.52it/s]


[P2] Epoch 3/10: train_f1=0.9040 | val_f1=0.6891
  Saved best model (val_f1=0.6891)


... (10 outputs left out)

[P2] Epoch 9/10 [Val]: 100%|██████████| 88/88 [00:58<00:00,  1.52it/s]


[P2] Epoch 9/10: train_f1=0.9602 | val_f1=0.7394


[P2] Epoch 10/10 [Val]: 100%|██████████| 88/88 [00:57<00:00,  1.53it/s]


[P2] Epoch 10/10: train_f1=0.9653 | val_f1=0.7531
  Saved best model (val_f1=0.7531)


## The End